# 🎨 Qwen-Image-Edit-2509 Interactive Demo

This notebook provides an interactive GUI for experimenting with the Qwen-Image-Edit-2509 model.

**Features:**
- Upload base image and optional reference image
- Enter custom prompts
- Adjust inference parameters (steps, seed, CFG scale)
- View results with timing information
- Auto-save all generated images

**Workflow:**
1. Run Cell 1 to set up dependencies and configuration
2. Run Cell 2 to load the model (takes ~2 minutes)
3. Run Cell 3 to launch the interactive GUI

In [ ]:
"""
=============================================================================
CELL 1: Setup & Configuration
=============================================================================
Run this cell first to import libraries and set up configuration.
"""

import os
import gc
import math
import time
import torch
from PIL import Image
from datetime import datetime
from diffusers import QwenImageEditPlusPipeline, FlowMatchEulerDiscreteScheduler

# Install gradio if not present
try:
    import gradio as gr
    print(f"✅ Gradio version: {gr.__version__}")
except ImportError:
    print("📦 Installing Gradio...")
    !pip install gradio -q
    import gradio as gr
    print(f"✅ Gradio installed: {gr.__version__}")

# =============================================================================
# CONFIGURATION - Modify these as needed
# =============================================================================

# Output directory for saved images
OUTPUT_DIR = "/workspace/wedding_decor/images/demo_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model settings
USE_LIGHTNING_LORA = True  # Set to False for full 40-step inference
LORA_WEIGHTS = "Qwen-Image-Edit-2509/Qwen-Image-Edit-2509-Lightning-8steps-V1.0-fp32.safetensors"

# Image dimensions (based on Qwen-Image-Edit-2509 optimal settings)
FIXED_WIDTH = 1024
FIXED_HEIGHT = 1024
REF_SIZE = 384  # Optimal for text encoder control

# Default generation settings
DEFAULT_STEPS = 8 if USE_LIGHTNING_LORA else 40
DEFAULT_SEED = 42
DEFAULT_CFG = 1.0  # true_cfg_scale

print(f"\n{'='*60}")
print("📋 Configuration")
print(f"{'='*60}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Lightning LoRA: {USE_LIGHTNING_LORA}")
print(f"Image size: {FIXED_WIDTH}x{FIXED_HEIGHT}")
print(f"Reference size: {REF_SIZE}x{REF_SIZE}")
print(f"Default steps: {DEFAULT_STEPS}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"{'='*60}")
print("\n✅ Cell 1 complete! Run Cell 2 to load the model.")

In [ ]:
"""
=============================================================================
CELL 2: Load Model
=============================================================================
This cell loads the Qwen-Image-Edit-2509 model and optional Lightning LoRA.
Takes approximately 2 minutes on first run.
"""

print(f"\n{'='*60}")
print("🚀 Loading Qwen-Image-Edit-2509 Model")
print(f"{'='*60}\n")

# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()

load_start = time.time()

# Scheduler configuration with dynamic shifting
scheduler_config = {
    "base_image_seq_len": 256,
    "base_shift": math.log(3),
    "invert_sigmas": False,
    "max_image_seq_len": 8192,
    "max_shift": math.log(3),
    "num_train_timesteps": 1000,
    "shift": 1.0,
    "shift_terminal": None,
    "stochastic_sampling": False,
    "time_shift_type": "exponential",
    "use_beta_sigmas": False,
    "use_dynamic_shifting": True,
    "use_exponential_sigmas": False,
    "use_karras_sigmas": False,
}
scheduler = FlowMatchEulerDiscreteScheduler.from_config(scheduler_config)

# Load pipeline
print("📦 Loading base model...")
pipeline = QwenImageEditPlusPipeline.from_pretrained(
    "Qwen/Qwen-Image-Edit-2509",
    scheduler=scheduler,
    torch_dtype=torch.bfloat16,
).to("cuda")

print(f"✅ Base model loaded in {time.time() - load_start:.1f}s")

# Load Lightning LoRA for faster inference
if USE_LIGHTNING_LORA:
    print("\n⚡ Loading Lightning 8-step LoRA...")
    pipeline.load_lora_weights(
        "lightx2v/Qwen-Image-Lightning",
        weight_name=LORA_WEIGHTS
    )
    print("✅ LoRA loaded")

# Disable progress bar for cleaner output
pipeline.set_progress_bar_config(disable=True)

# Warmup run
print("\n🔥 Running warmup inference...")
warmup_start = time.time()
dummy = Image.new('RGB', (FIXED_WIDTH, FIXED_HEIGHT), 'white')
dummy_ref = Image.new('RGB', (REF_SIZE, REF_SIZE), 'gray')

with torch.inference_mode():
    _ = pipeline(
        image=[dummy, dummy_ref],
        prompt="warmup",
        num_inference_steps=4,
        true_cfg_scale=1.0,
        guidance_scale=1.0,
    )

gc.collect()
torch.cuda.empty_cache()
print(f"✅ Warmup complete in {time.time() - warmup_start:.1f}s")

total_load_time = time.time() - load_start
print(f"\n{'='*60}")
print(f"✅ Model ready! Total load time: {total_load_time:.1f}s")
print(f"{'='*60}")
print("\n✅ Cell 2 complete! Run Cell 3 to launch the GUI.")

In [ ]:
"""
=============================================================================
CELL 3: Interactive GUI
=============================================================================
Launch the Gradio interface for interactive image editing.
The GUI will run indefinitely until you stop it or restart the kernel.
"""

# =============================================================================
# Helper Functions
# =============================================================================

def resize_to_fixed(img: Image.Image) -> Image.Image:
    """Resize image to fixed output dimensions"""
    return img.resize((FIXED_WIDTH, FIXED_HEIGHT), Image.LANCZOS)


def resize_reference(img: Image.Image) -> Image.Image:
    """Resize reference image to optimal control size (384x384)"""
    w, h = img.size
    if w != h:
        # Center crop to square
        min_dim = min(w, h)
        left = (w - min_dim) // 2
        top = (h - min_dim) // 2
        img = img.crop((left, top, left + min_dim, top + min_dim))
    return img.resize((REF_SIZE, REF_SIZE), Image.LANCZOS)


def format_time(seconds: float) -> str:
    """Format time for display"""
    if seconds < 60:
        return f"{seconds:.2f}s"
    return f"{int(seconds // 60)}m {seconds % 60:.1f}s"


# Generation counter for unique filenames
generation_counter = 0


def generate_image(
    base_image,
    reference_image,
    prompt,
    negative_prompt,
    num_steps,
    seed,
    true_cfg_scale,
    auto_save
):
    """
    Main generation function called by Gradio interface.
    """
    global generation_counter
    
    # Validate inputs
    if base_image is None:
        return None, "❌ Error: Please upload a base image."
    
    if not prompt or prompt.strip() == "":
        return None, "❌ Error: Please enter a prompt."
    
    try:
        start_time = time.time()
        
        # Process base image
        if isinstance(base_image, str):
            base_img = Image.open(base_image).convert("RGB")
        else:
            base_img = Image.fromarray(base_image).convert("RGB")
        
        original_size = base_img.size
        base_img = resize_to_fixed(base_img)
        
        # Process reference image (or use blank if not provided)
        if reference_image is not None:
            if isinstance(reference_image, str):
                ref_img = Image.open(reference_image).convert("RGB")
            else:
                ref_img = Image.fromarray(reference_image).convert("RGB")
            ref_img = resize_reference(ref_img)
            ref_info = f"{ref_img.size[0]}x{ref_img.size[1]}"
        else:
            ref_img = Image.new('RGB', (REF_SIZE, REF_SIZE), (250, 250, 250))
            ref_info = "None (blank)"
        
        preprocess_time = time.time() - start_time
        
        # Run inference
        inference_start = time.time()
        
        with torch.inference_mode():
            output = pipeline(
                image=[base_img, ref_img],
                prompt=prompt,
                negative_prompt=negative_prompt if negative_prompt else None,
                num_inference_steps=int(num_steps),
                true_cfg_scale=float(true_cfg_scale),
                guidance_scale=1.0,
                generator=torch.Generator("cuda").manual_seed(int(seed)),
            )
        
        inference_time = time.time() - inference_start
        result = output.images[0]
        
        # Ensure correct size
        if result.size != (FIXED_WIDTH, FIXED_HEIGHT):
            result = resize_to_fixed(result)
        
        total_time = time.time() - start_time
        
        # Auto-save if enabled
        save_info = ""
        if auto_save:
            generation_counter += 1
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"gen_{generation_counter:04d}_{timestamp}.png"
            save_path = os.path.join(OUTPUT_DIR, filename)
            result.save(save_path)
            save_info = f"\n💾 Saved: {filename}"
        
        # Build status message
        status = f"""
✅ Generation Complete!
{'='*40}
⏱️  Total time: {format_time(total_time)}
   - Preprocessing: {format_time(preprocess_time)}
   - Inference: {format_time(inference_time)}

📊 Settings:
   - Steps: {int(num_steps)}
   - Seed: {int(seed)}
   - CFG Scale: {true_cfg_scale}
   - Base image: {original_size[0]}x{original_size[1]} → {FIXED_WIDTH}x{FIXED_HEIGHT}
   - Reference: {ref_info}{save_info}
"""
        
        # Clear GPU cache
        gc.collect()
        torch.cuda.empty_cache()
        
        return result, status
    
    except Exception as e:
        import traceback
        error_msg = f"❌ Error: {str(e)}\n\n{traceback.format_exc()}"
        return None, error_msg


# =============================================================================
# Build Gradio Interface
# =============================================================================

# Custom CSS for better appearance
custom_css = """
.gradio-container {
    max-width: 1400px !important;
}
#output-image {
    min-height: 512px;
}
"""

with gr.Blocks(css=custom_css, title="Qwen-Image-Edit Demo") as demo:
    gr.Markdown("""
    # 🎨 Qwen-Image-Edit-2509 Interactive Demo
    
    Upload a base image, optionally add a reference image, enter your prompt, and click Generate!
    
    **Tips:**
    - Reference image guides the style/appearance of new elements
    - Leave reference empty for prompt-only edits
    - Use negative prompts to avoid unwanted elements
    - Lower steps (4-8) with Lightning LoRA, higher (30-50) without
    """)
    
    with gr.Row():
        # Left column - Inputs
        with gr.Column(scale=1):
            gr.Markdown("### 📥 Inputs")
            
            base_image = gr.Image(
                label="Base Image (Required)",
                type="numpy",
                height=300,
            )
            
            reference_image = gr.Image(
                label="Reference Image (Optional)",
                type="numpy",
                height=200,
            )
            
            prompt = gr.Textbox(
                label="Prompt",
                placeholder="Describe what you want to change or add...",
                lines=3,
            )
            
            negative_prompt = gr.Textbox(
                label="Negative Prompt (Optional)",
                placeholder="What to avoid...",
                lines=2,
                value="blurry, distorted, low quality, deformed, artifacts",
            )
            
            with gr.Row():
                num_steps = gr.Slider(
                    minimum=1,
                    maximum=50,
                    value=DEFAULT_STEPS,
                    step=1,
                    label="Inference Steps",
                )
                seed = gr.Number(
                    value=DEFAULT_SEED,
                    label="Seed",
                    precision=0,
                )
            
            with gr.Row():
                true_cfg_scale = gr.Slider(
                    minimum=1.0,
                    maximum=10.0,
                    value=DEFAULT_CFG,
                    step=0.5,
                    label="CFG Scale (1.0 for Lightning LoRA)",
                )
                auto_save = gr.Checkbox(
                    value=True,
                    label="Auto-save results",
                )
            
            generate_btn = gr.Button(
                "🚀 Generate",
                variant="primary",
                size="lg",
            )
        
        # Right column - Output
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Output")
            
            output_image = gr.Image(
                label="Generated Image",
                type="pil",
                height=512,
                elem_id="output-image",
            )
            
            status_text = gr.Textbox(
                label="Status & Timing",
                lines=12,
                interactive=False,
            )
    
    # Example prompts
    gr.Markdown("### 💡 Example Prompts")
    gr.Examples(
        examples=[
            ["Change the tablecloth to a royal blue pintuck tablecloth."],
            ["Replace all chairs with clear acrylic ghost chiavari chairs."],
            ["Add 8 gold charger plates at each place setting on the table."],
            ["Add elegant champagne flutes at each place setting."],
            ["Add a green crystal vase centerpiece in the center of the table."],
            ["Remove the table from this image. Leave all items floating in the air."],
        ],
        inputs=[prompt],
        label="Click to use example prompt",
    )
    
    # Connect the generate button
    generate_btn.click(
        fn=generate_image,
        inputs=[
            base_image,
            reference_image,
            prompt,
            negative_prompt,
            num_steps,
            seed,
            true_cfg_scale,
            auto_save,
        ],
        outputs=[output_image, status_text],
    )

# =============================================================================
# Launch the GUI
# =============================================================================

print(f"\n{'='*60}")
print("🚀 Launching Gradio Interface")
print(f"{'='*60}")
print(f"\n📁 Results will be saved to: {OUTPUT_DIR}")
print("\n⚠️  To stop the GUI, interrupt/restart the kernel.")
print(f"{'='*60}\n")

# Launch with share=True if you want a public URL
demo.launch(
    share=False,  # Set to True for public URL
    server_name="0.0.0.0",  # Allow external connections
    server_port=7860,
    show_error=True,
)

---

## 📝 Notes

### Image Sizes (Based on Qwen-Image-Edit-2509 Research)
- **Base/Output images**: 1024x1024 (max ~1 megapixel supported)
- **Reference images**: 384x384 (optimal for text encoder control)

### Tips for Best Results
1. **With Reference Image**: The model will try to match the style/appearance of the reference
2. **Without Reference**: Relies purely on the text prompt
3. **Table Removal**: Use prompt like "Remove the table, leave items floating"
4. **Fusion**: Upload floating items as reference to combine with base scene

### Saved Images
All generated images (when auto-save is enabled) are saved to:
```
/workspace/wedding_decor/images/demo_outputs/
```

### Stopping the GUI
- Click the **Stop** button in Jupyter, or
- Restart the kernel